# Part A — HR-only population model (LOSO)

**Model identity**

- LOSO ensemble weights: XGB 0.45 / ANN 0.55
- ANN output: train-only min–max each fold; subject-grouped early stopping (`GroupShuffleSplit`)
- Personal Part B scores use the **deployed** offline weights 0.6 / 0.4 (`p_python_art_cfg`). Those are not LOSO fold predictions.
- The export cell below is a **third** training run for packaging only. It is not an evaluation result and must not overwrite phone artifacts used by Part B.
- HR vs HR+EDA is an **XGBoost-only** ablation on the same LOSO subjects (N ≈ 0.57 vs 0.53). Not the ensemble. Not Vos et al. SynthesizedStressData / held-out WESAD. StressData has four EDA columns.

## Setup

In [ ]:
!pip -q install xgboost scikit-learn pandas numpy matplotlib seaborn tensorflow

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = Path('/content/drive/MyDrive/cloudnine-stress-ml')
except Exception:
    PROJECT = Path.cwd()
    if not (PROJECT / 'docs').exists() and (PROJECT.parent / 'docs').exists():
        PROJECT = PROJECT.parent

DATA_DIR = PROJECT / 'data'
# Packaged models for optional export only — never the phone/Part B artifact tree
EXPORT_DIR = PROJECT / 'part_a_export_scratch'
OUTPUT_DIR = PROJECT / 'part_a_results'
LOCAL_ARTIFACTS = PROJECT / 'docs' / 'Cloudnine Stress Artifacts'
PHONE_ARTIFACTS = LOCAL_ARTIFACTS  # read-only for Part B; do not write here from this notebook

for p in (DATA_DIR, EXPORT_DIR, OUTPUT_DIR):
    p.mkdir(parents=True, exist_ok=True)

DATA_CSV = None
for cand in [LOCAL_ARTIFACTS / 'StressData.csv', DATA_DIR / 'StressData.csv']:
    if cand.exists():
        DATA_CSV = cand
        break
if DATA_CSV is None:
    DATA_CSV = DATA_DIR / 'StressData.csv'

LOSO_CSV = LOCAL_ARTIFACTS / 'revision' / 'loso_results.csv'
print('DATA_CSV', DATA_CSV.exists(), DATA_CSV)
print('LOSO_CSV', LOSO_CSV.exists(), LOSO_CSV)
print('EXPORT_DIR (scratch only)', EXPORT_DIR)
print('PHONE_ARTIFACTS (do not overwrite)', PHONE_ARTIFACTS)

## Data

In [ ]:
import zipfile
import urllib.request

STRESSDATA_ZIP = 'https://github.com/xalentis/Stress/raw/master/StressData.zip'
if not DATA_CSV.exists():
    zip_path = DATA_CSV.with_suffix('.zip')
    urllib.request.urlretrieve(STRESSDATA_ZIP, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        member = next(m for m in zf.namelist() if m.endswith('StressData.csv'))
        zf.extract(member, DATA_CSV.parent)
        extracted = DATA_CSV.parent / member
        if extracted != DATA_CSV:
            extracted.replace(DATA_CSV)
print('ok', DATA_CSV)

In [ ]:
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix, matthews_corrcoef,
)

import xgboost as xgb
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

FEATURES = ['hrrange', 'hrvar', 'hrstd', 'hrmin', 'hrmax', 'hrkurt']
LABEL = 'metric'
SUBJECT_COL = 'Subject'

RUN_FULL_LOSO = False  # True = full ~99-subject retrain; False = reload revision/loso_results.csv
RANDOM_SEED = 123

XGB_PARAMS = {
    'eta': 0.5, 'max_depth': 8, 'subsample': 0.70, 'colsample_bytree': 0.8,
    'objective': 'binary:logistic', 'eval_metric': 'logloss', 'seed': RANDOM_SEED,
}
ANN_EPOCHS, ANN_BATCH, ANN_PATIENCE = 120, 512, 3
XGB_WEIGHT, ANN_WEIGHT = 0.45, 0.55

np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)
sns.set_theme(style='whitegrid', context='notebook')
print('TF', tf.__version__)

In [ ]:
df = pd.read_csv(DATA_CSV)
df.columns = [c.strip() for c in df.columns]
if SUBJECT_COL not in df.columns:
    SUBJECT_COL = next(c for c in df.columns if c.lower() == 'subject')
missing = [c for c in FEATURES + [LABEL, SUBJECT_COL] if c not in df.columns]
if missing:
    raise KeyError(missing)
data = df.dropna(subset=FEATURES + [LABEL, SUBJECT_COL]).copy()
data[LABEL] = data[LABEL].astype(int)
data[SUBJECT_COL] = data[SUBJECT_COL].astype(str)
eda_cols = [c for c in data.columns if 'eda' in c.lower()]
print(data.shape, 'subjects', data[SUBJECT_COL].nunique(), 'EDA cols', len(eda_cols), eda_cols)
data[SUBJECT_COL].str[0].value_counts()

## Models

In [ ]:
def scale_pos_weight(y):
    y = np.asarray(y)
    return float((y == 0).sum()) / max(float((y == 1).sum()), 1.0)


def build_ann(input_dim):
    hidden = max(input_dim // 2, 1)
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(input_dim, activation='relu'),
        layers.Dense(hidden, activation='relu'),
        layers.Dense(1, activation='linear', name='stress_linear'),
    ])
    model.compile(optimizer=keras.optimizers.Adamax(), loss='mse')
    return model


def minmax_norm_fixed(arr, lo, hi):
    arr = np.asarray(arr, float)
    if hi - lo < 1e-9:
        return np.zeros_like(arr)
    return np.clip((arr - lo) / (hi - lo), 0.0, 1.0)


def metrics_full(y_true, p, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    p = np.asarray(p, float)
    y_hat = (p >= thr).astype(int)
    out = {
        'accuracy': float(accuracy_score(y_true, y_hat)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_hat)),
        'precision': float(precision_score(y_true, y_hat, zero_division=0)),
        'recall': float(recall_score(y_true, y_hat, zero_division=0)),
        'f1': float(f1_score(y_true, y_hat, zero_division=0)),
        'macro_f1': float(f1_score(y_true, y_hat, average='macro', zero_division=0)),
        'mcc': float(matthews_corrcoef(y_true, y_hat)) if len(np.unique(y_true)) > 1 else np.nan,
        'auc': float(roc_auc_score(y_true, p)) if len(np.unique(y_true)) > 1 else np.nan,
        'n': int(len(y_true)),
        'prevalence': float(y_true.mean()),
        'auc_defined': bool(len(np.unique(y_true)) > 1),
    }
    cm = confusion_matrix(y_true, y_hat, labels=[0, 1])
    out['tn'], out['fp'], out['fn'], out['tp'] = map(int, cm.ravel())
    return out


def _inner_split(X, y, groups, seed):
    if len(np.unique(groups)) >= 3:
        gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
        return next(gss.split(X, y, groups))
    fake = np.arange(len(y)) // max(len(y) // 5, 1)
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    return next(gss.split(X, y, fake))


def train_fold(X_train, y_train, groups_train, X_test, seed):
    tr_idx, va_idx = _inner_split(X_train, y_train, groups_train, seed)
    params = XGB_PARAMS.copy()
    params['seed'] = seed
    params['scale_pos_weight'] = scale_pos_weight(y_train[tr_idx])
    dtr = xgb.DMatrix(X_train[tr_idx], label=y_train[tr_idx], feature_names=FEATURES)
    dva = xgb.DMatrix(X_train[va_idx], label=y_train[va_idx], feature_names=FEATURES)
    xgb_es = xgb.train(
        params, dtr, num_boost_round=500,
        evals=[(dtr, 'train'), (dva, 'val')],
        early_stopping_rounds=3, verbose_eval=False,
    )
    best_rounds = max(int(xgb_es.best_iteration) + 1, 1)
    params_full = XGB_PARAMS.copy()
    params_full['seed'] = seed
    params_full['scale_pos_weight'] = scale_pos_weight(y_train)
    xgb_model = xgb.train(
        params_full,
        xgb.DMatrix(X_train, label=y_train, feature_names=FEATURES),
        num_boost_round=best_rounds, verbose_eval=False,
    )
    xgb_p = xgb_model.predict(xgb.DMatrix(X_test, feature_names=FEATURES))

    scaler_es = StandardScaler()
    X_tr_s = scaler_es.fit_transform(X_train[tr_idx])
    X_va_s = scaler_es.transform(X_train[va_idx])
    tf.keras.utils.set_random_seed(seed)
    ann_es = build_ann(X_train.shape[1])
    hist = ann_es.fit(
        X_tr_s, y_train[tr_idx],
        validation_data=(X_va_s, y_train[va_idx]),
        epochs=ANN_EPOCHS, batch_size=ANN_BATCH,
        callbacks=[keras.callbacks.EarlyStopping(
            monitor='val_loss', patience=ANN_PATIENCE, restore_best_weights=True,
        )],
        verbose=0,
    )
    best_epoch = max(int(np.argmin(hist.history['val_loss'])) + 1, 1)

    scaler = StandardScaler().fit(X_train)
    X_train_s = scaler.transform(X_train)
    X_test_s = scaler.transform(X_test)
    tf.keras.utils.set_random_seed(seed)
    ann = build_ann(X_train.shape[1])
    ann.fit(X_train_s, y_train, epochs=best_epoch, batch_size=ANN_BATCH, verbose=0)

    ann_raw_tr = ann.predict(X_train_s, verbose=0).ravel()
    ann_raw_te = ann.predict(X_test_s, verbose=0).ravel()
    lo, hi = float(ann_raw_tr.min()), float(ann_raw_tr.max())
    ann_train_only = minmax_norm_fixed(ann_raw_te, lo, hi)
    ann_batch = minmax_norm_fixed(ann_raw_te, float(ann_raw_te.min()), float(ann_raw_te.max()))
    return {
        'ens_train_only': XGB_WEIGHT * xgb_p + ANN_WEIGHT * ann_train_only,
        'ens_batch': XGB_WEIGHT * xgb_p + ANN_WEIGHT * ann_batch,
        'xgb_rounds': best_rounds, 'ann_epochs': best_epoch,
    }

## LOSO (train or reload)

In [ ]:
# Default: re-aggregate existing per-subject file. Set RUN_FULL_LOSO=True only to retrain all subjects.
rev = LOCAL_ARTIFACTS / 'revision'
rev.mkdir(parents=True, exist_ok=True)

if RUN_FULL_LOSO:
    subjects = sorted(data[SUBJECT_COL].unique())
    print(f'FULL LOSO: {len(subjects)} subjects')
    rows = []
    for i, subject in enumerate(subjects, 1):
        train = data[data[SUBJECT_COL] != subject]
        test = data[data[SUBJECT_COL] == subject]
        pred = train_fold(
            train[FEATURES].astype(float).values,
            train[LABEL].astype(int).values,
            train[SUBJECT_COL].values,
            test[FEATURES].astype(float).values,
            seed=RANDOM_SEED + i,
        )
        y_te = test[LABEL].astype(int).values
        y_tr = train[LABEL].astype(int).values
        train_maj = 1 if y_tr.mean() >= 0.5 else 0
        maj_acc = float((y_te == train_maj).mean())
        for protocol, p in [
            ('train_only_ann_minmax', pred['ens_train_only']),
            ('batch_minmax_heldout_subject', pred['ens_batch']),
        ]:
            m = metrics_full(y_te, p)
            m['majority_train_acc'] = maj_acc
            rows.append({
                'subject': subject, 'cohort': str(subject)[0].upper(),
                'protocol': protocol, 'xgb_rounds': pred['xgb_rounds'],
                'ann_epochs': pred['ann_epochs'], **m,
            })
        print(f'[{i}/{len(subjects)}] {subject}')
    loso_df = pd.DataFrame(rows)
    loso_df.to_csv(OUTPUT_DIR / 'loso_results.csv', index=False)
    loso_df.to_csv(rev / 'loso_results.csv', index=False)
elif LOSO_CSV.exists():
    loso_df = pd.read_csv(LOSO_CSV)
    print('Loaded', LOSO_CSV, 'rows', len(loso_df))
    n_subj = loso_df.loc[loso_df.protocol == 'train_only_ann_minmax', 'subject'].nunique()
    if n_subj < 90:
        raise RuntimeError(
            f'Expected full LOSO (~99 subjects), found {n_subj}. '
            'Set RUN_FULL_LOSO=True to retrain, or restore revision/loso_results.csv.'
        )
else:
    raise FileNotFoundError(
        f'Missing {LOSO_CSV}. Place full loso_results.csv under revision/, or set RUN_FULL_LOSO=True.'
    )

primary = loso_df[loso_df.protocol == 'train_only_ann_minmax'].copy()
print(primary.groupby('cohort').size())


## Honest cohort averages

In [ ]:
def summarize_honest(df, protocol='train_only_ann_minmax', n_boot=5000, seed=RANDOM_SEED):
    # n_subjects for a metric = count of subjects with a defined value (esp. AUC).
    sub_all = df[df.protocol == protocol].copy()
    metrics = [
        'accuracy', 'precision', 'recall', 'balanced_accuracy', 'f1', 'auc',
        'majority_train_acc',
    ]
    rows = []
    for scope in ['N', 'S', 'W', 'U', 'all']:
        sub = sub_all if scope == 'all' else sub_all[sub_all.cohort == scope]
        if len(sub) == 0:
            continue
        n_one_class = int((~sub.get('auc_defined', sub['auc'].notna())).sum()) if 'auc_defined' in sub.columns else int(sub['auc'].isna().sum())
        for metric in metrics:
            vals = sub[metric].dropna().to_numpy()
            n_def = int(len(vals))
            if n_def == 0:
                rows.append({
                    'protocol': protocol, 'scope': scope,
                    'n_subjects_in_scope': int(len(sub)),
                    'n_defined': 0, 'metric': metric,
                    'mean': np.nan, 'ci95_lo': np.nan, 'ci95_hi': np.nan,
                    'note': 'no_defined_values',
                })
                continue
            rng = np.random.default_rng(seed)
            boots = [vals[rng.integers(0, n_def, n_def)].mean() for _ in range(n_boot)]
            note = ''
            if metric == 'auc' and scope in ('U', 'all'):
                note = f'auc_mean_over_{n_def}_of_{len(sub)}_subjects'
            if scope == 'U' and metric == 'accuracy':
                note = f'{int(sub["auc"].isna().sum())}_of_{len(sub)}_U_subjects_one_class_or_auc_undefined'
            rows.append({
                'protocol': protocol, 'scope': scope,
                'n_subjects_in_scope': int(len(sub)),
                'n_defined': n_def, 'metric': metric,
                'mean': float(np.mean(vals)),
                'ci95_lo': float(np.percentile(boots, 2.5)),
                'ci95_hi': float(np.percentile(boots, 97.5)),
                'note': note,
            })
    return pd.DataFrame(rows)

summary = summarize_honest(loso_df)
out = LOCAL_ARTIFACTS / 'revision'
out.mkdir(parents=True, exist_ok=True)
summary.to_csv(out / 'loso_summary.csv', index=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
summary.to_csv(OUTPUT_DIR / 'loso_summary.csv', index=False)

print('Primary scope = N (train_only):')
print(summary[(summary.scope == 'N') & (summary.protocol == 'train_only_ann_minmax')][
    ['metric', 'n_defined', 'mean', 'ci95_lo', 'ci95_hi']
].to_string(index=False))
print('\nAll-subjects AUC uses n_defined, not 99:')
print(summary[(summary.scope == 'all') & (summary.metric == 'auc')].to_string(index=False))
u = primary[primary.cohort == 'U']
print('U subjects:', len(u), '; AUC undefined:', int(u['auc'].isna().sum()))

## XGB ablation: HR vs HR+EDA

In [ ]:
# Prefer existing ablation CSV if present; else run XGB-only LOSO.
ABL_PATH = LOCAL_ARTIFACTS / 'revision' / 'part_a_xgb_hr_vs_eda_summary.csv'
ABL_PER = LOCAL_ARTIFACTS / 'revision' / 'part_a_xgb_loso_per_subject.csv'

def train_xgb_only(X_train, y_train, groups_train, X_test, seed):
    tr_idx, va_idx = _inner_split(X_train, y_train, groups_train, seed)
    params = XGB_PARAMS.copy()
    params['seed'] = seed
    params['scale_pos_weight'] = scale_pos_weight(y_train[tr_idx])
    model = xgb.train(
        params,
        xgb.DMatrix(X_train[tr_idx], label=y_train[tr_idx]),
        num_boost_round=500,
        evals=[
            (xgb.DMatrix(X_train[tr_idx], label=y_train[tr_idx]), 'train'),
            (xgb.DMatrix(X_train[va_idx], label=y_train[va_idx]), 'val'),
        ],
        early_stopping_rounds=3, verbose_eval=False,
    )
    best = max(int(model.best_iteration) + 1, 1)
    params['scale_pos_weight'] = scale_pos_weight(y_train)
    model2 = xgb.train(
        params, xgb.DMatrix(X_train, label=y_train),
        num_boost_round=best, verbose_eval=False,
    )
    return model2.predict(xgb.DMatrix(X_test))

if ABL_PATH.exists():
    abl_sum = pd.read_csv(ABL_PATH)
    print('Loaded', ABL_PATH)
    print(abl_sum[(abl_sum.scope == 'N') & (abl_sum.metric.isin(['accuracy', 'auc', 'f1']))].to_string(index=False))
else:
    print('EDA cols', eda_cols)
    abl_rows = []
    feat_sets = [('HR_only', FEATURES)]
    if eda_cols:
        feat_sets.append(('HR_plus_EDA', FEATURES + eda_cols))
    for tag, cols in feat_sets:
        d = data.dropna(subset=cols).copy()
        for i, subject in enumerate(sorted(d[SUBJECT_COL].unique()), 1):
            train = d[d[SUBJECT_COL] != subject]
            test = d[d[SUBJECT_COL] == subject]
            p = train_xgb_only(
                train[cols].astype(float).values,
                train[LABEL].astype(int).values,
                train[SUBJECT_COL].values,
                test[cols].astype(float).values,
                RANDOM_SEED + i,
            )
            m = metrics_full(test[LABEL].astype(int).values, p)
            train_maj = 1 if train[LABEL].mean() >= 0.5 else 0
            m['majority_train_acc'] = float((test[LABEL].astype(int).values == train_maj).mean())
            abl_rows.append({
                'subject': subject, 'feature_set': tag,
                'cohort': str(subject)[0].upper(), **m,
            })
        print(tag, 'done')
    abl = pd.DataFrame(abl_rows)
    abl.to_csv(LOCAL_ARTIFACTS / 'revision' / 'part_a_xgb_loso_per_subject.csv', index=False)
    # quick N means
    for tag in abl.feature_set.unique():
        sub = abl[(abl.feature_set == tag) & (abl.cohort == 'N')]
        print(tag, 'N', sub[['accuracy', 'f1', 'auc']].mean().round(4).to_dict())

## Export scratch (not evaluation; does not touch phone artifacts)

In [ ]:
# Third training run for packaging only. Writes to EXPORT_DIR, never to PHONE_ARTIFACTS.
assert EXPORT_DIR.resolve() != PHONE_ARTIFACTS.resolve()
X_all = data[FEATURES].astype(float).values
y_all = data[LABEL].astype(int).values
X_tr, X_te, y_tr, y_te = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_SEED, stratify=y_all,
)

params = XGB_PARAMS.copy()
params['scale_pos_weight'] = scale_pos_weight(y_tr)
dtr = xgb.DMatrix(X_tr, label=y_tr, feature_names=FEATURES)
dva = xgb.DMatrix(X_te, label=y_te, feature_names=FEATURES)
xgb_es = xgb.train(
    params, dtr, num_boost_round=500,
    evals=[(dtr, 'train'), (dva, 'val')],
    early_stopping_rounds=3, verbose_eval=False,
)
best_rounds = max(int(xgb_es.best_iteration) + 1, 1)
final_xgb = xgb.train(
    params, xgb.DMatrix(X_all, label=y_all, feature_names=FEATURES),
    num_boost_round=best_rounds, verbose_eval=False,
)

scaler = StandardScaler().fit(X_tr)
ann = build_ann(len(FEATURES))
ann.fit(
    scaler.transform(X_tr), y_tr,
    validation_data=(scaler.transform(X_te), y_te),
    epochs=ANN_EPOCHS, batch_size=ANN_BATCH,
    callbacks=[keras.callbacks.EarlyStopping(
        monitor='val_loss', patience=ANN_PATIENCE, restore_best_weights=True,
    )],
    verbose=0,
)
ann_raw_tr = ann.predict(scaler.transform(X_tr), verbose=0).ravel()
ANN_OUTPUT_MIN, ANN_OUTPUT_MAX = float(ann_raw_tr.min()), float(ann_raw_tr.max())

final_xgb.save_model(EXPORT_DIR / 'stress_xgb.json')
ann.save(EXPORT_DIR / 'stress_ann.keras')
json.dump(
    {'mean': scaler.mean_.tolist(), 'std': scaler.scale_.tolist()},
    open(EXPORT_DIR / 'ann_scaler.json', 'w'), indent=2,
)
json.dump(
    {'ann_output_min': ANN_OUTPUT_MIN, 'ann_output_max': ANN_OUTPUT_MAX},
    open(EXPORT_DIR / 'ann_minmax.json', 'w'), indent=2,
)
json.dump(
    {
        'xgb_weight': 0.6, 'ann_weight': 0.4,
        'ann_output_min': ANN_OUTPUT_MIN, 'ann_output_max': ANN_OUTPUT_MAX,
        'note': 'export scratch only; Part B uses frozen phone/offline p_python_art_cfg',
        'not_evaluation': True,
    },
    open(EXPORT_DIR / 'ensemble_config.json', 'w'), indent=2,
)
print('Wrote export scratch to', EXPORT_DIR)
print('Phone artifacts untouched at', PHONE_ARTIFACTS)